# BBPLL Performance: 100 MHz Reference → 8 GHz Output (N = 80)

Companion to `BBPLL_BW_PM_wrp_latency.ipynb`, which models the **same 8 GHz output** from a
**1 GHz reference with N = 8**. Here the reference drops 10× to **100 MHz** and the divider
ratio rises to **N = 80**.

This is not a parameter swap — dropping the reference changes the loop qualitatively:

| | 1 GHz ref, N=8 | 100 MHz ref, N=80 |
|---|---|---|
| Sampling period | 1 ns | **10 ns** |
| Usable loop BW | 150 MHz (fref/6.7) | **~10 MHz** (15× lower) |
| K<sub>bbpd</sub> for same abs. jitter | 508 | **5079** (σ<sub>rad</sub> = σ<sub>t</sub>·2π·f<sub>ref</sub>) |
| BBPD quantization floor | −91.8 dBc/Hz | **−81.8 dBc/Hz** (+10 dB) |
| DAC / DSM quantization floor | −100.8 dBc/Hz | **−90.8 dBc/Hz** (+10 dB) |
| Reference noise × N² | +18.1 dB | **+38.1 dB** (+20 dB) |
| VCO phase noise @ 8 GHz | unchanged | unchanged |

The hardcoded `Ki = 0.159…` / `Kp = 1.483…` from the 1 GHz notebook are **invalid here** —
they were solved for that design point. This notebook re-solves them.

## Two traps this notebook is built around

**1. There is a phase-margin ceiling.** At a 10 ns sampling period the ZoH half-sample delay
alone costs −18° at 10 MHz, capping the achievable phase margin at **69.5° @ 10 MHz**
(73.6° @ 8 MHz, 65.4° @ 12 MHz). The old notebook's **PM = 77° is unreachable at any usable
bandwidth.** We use PM = 60°.

**2. `fsolve` diverges *silently* past that ceiling.** Asking for more phase margin than the
loop physically has does not raise — it returns `kp ~ 1e8` and a nonsense jitter of ~1e23 fs,
and `get_pm_bw` then reports a plausible-looking but bogus 348°. Every solve in this notebook is
therefore guarded (feasibility pre-check + convergence flag + residual + independent
`get_pm_bw` post-check).

## Headline result

**≈ 1.03 ps** RMS output jitter at BW = 10 MHz / PM = 60°, versus the ~200 fs class of the 1 GHz
design. No single term dominates: VCO 678 fs, reference 541 fs, BBPD quantization 504 fs,
DAC quantization 245 fs.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, so `bbpll` is importable

import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize
from matplotlib.ticker import FuncFormatter

import bbpll
from bbpll import components, noise
from bbpll import (PhaseNoise, get_jitter, get_pm_bw, get_shaped_phase_noise,
                   jitter_breakdown, jitter_breakdown_piechart,
                   metric_prefix_formatter, plot_phase_noise)

###################################################
# Frequency plan  --  THIS is what changed
###################################################
PLL_DIVIDER_RATIO = 80                              # was 8
PLL_FREF = 100e6                                    # was 1e9
PLL_FOUT = PLL_FREF * PLL_DIVIDER_RATIO             # 8 GHz, unchanged
PLL_DIGITAL_F = PLL_FREF                            # Digital frequency in Hz
PLL_DIGITAL_T = 1 / PLL_DIGITAL_F                   # 10 ns  (was 1 ns)

# Loop target. PM=77 (the 1 GHz value) is UNREACHABLE here -- see max_feasible_pm() below.
PLL_BW = 10e6                                       # was 150e6
PLL_PM = 60                                         # was 77

###################################################
# Hardware parameters -- deliberately UNCHANGED so
# that only the frequency plan differs
###################################################

# VCO
PLL_KVCO = 100e6 * 2 * np.pi                        # VCO gain in rad^-1/V
PLL_KDCO = 1.25e6 * 2 * np.pi                       # DCO gain in rad^-1/V
PLL_VCO_WHITE_100MHz = -130                         # [dBc/Hz] at 100 MHz offset
PLL_VCO_FLICKER_10kHz = -15                         # [dBc/Hz] at 10 kHz offset

# PFD (defined for a later hybrid analog/digital extension; unused in the digital-only loop)
PFD_GAIN = 1 / (2 * np.pi)                          # PFD gain in 1/rad
R = 1e3                                             # Loop filter resistance in Ohms

# Latencies
PLL_DLF_LATENCY_SECONDS = 521e-12
PLL_DIVIDER_LATENCY_SECONDS = 170e-12

# Delta-Sigma Modulator on the loop filter output. NOTE: DSM_FREQ tracks fref,
# so its quantization floor rises 10 dB relative to the 1 GHz design.
DSM_ORDER = 1
DSM_FREQ = 2 * PLL_FREF
DSM_T = 1 / DSM_FREQ

# Loop angular frequency, used for time <-> phase conversion at the BBPD input
PLL_ANGULAR_FREQUENCY = 2 * np.pi * PLL_FREF

# NOTE: Ki / Kp are deliberately NOT defined here. The 1 GHz values
# (Ki=0.15924771764231094, Kp=1.483004995346433) are invalid at this design
# point; they are re-solved in the "Solver" cell below.

f_offset = np.logspace(3, 9.6, 100000)              # 1 kHz .. ~4 GHz
INTEGRATION_BANDWIDTH = (1e3, 4e9)                  # 4 GHz = fout/2, still correct at 8 GHz out

print(f"Reference      : {PLL_FREF/1e6:.0f} MHz")
print(f"Divider ratio  : {PLL_DIVIDER_RATIO}")
print(f"Output         : {PLL_FOUT/1e9:.1f} GHz")
print(f"Sampling period: {PLL_DIGITAL_T*1e9:.0f} ns")
print(f"Target BW / PM : {PLL_BW/1e6:.0f} MHz / {PLL_PM}deg")
print(f"Ref noise x N^2: +{20*np.log10(PLL_DIVIDER_RATIO):.1f} dB "
      f"(vs +{20*np.log10(8):.1f} dB at N=8)")

In [ ]:
# Component transfer functions

# PFD -- kept for a future hybrid analog/digital Kp split; not used in the digital-only loop gain
pfd = components.PhaseFrequencyDetector(kpd=PFD_GAIN, input_referred_jitter=0)
tf_PFD = pfd.tf(f_offset)

# DVCO
dvco_phase_noise_obj = noise.PhaseNoise(
    white_noise_100MHz=PLL_VCO_WHITE_100MHz,
    flicker_noise_10kHz=PLL_VCO_FLICKER_10kHz,
)
dvco_phase_noise = dvco_phase_noise_obj.get_noise_power(f_offset)

dvco = components.DVCO(
    kvco=PLL_KVCO,
    kdco=PLL_KDCO,
    phase_noise=dvco_phase_noise,
)
tf_dvco_analog, tf_dvco_digital = dvco.tf(f_offset)

# Divider
divider = components.Divider(
    divider_ratio=PLL_DIVIDER_RATIO,
    latency=PLL_DIVIDER_LATENCY_SECONDS,
)
tf_divider = divider.tf(f_offset)

# The DLF is NOT built here -- its Ki/Kp are unknown until the solver runs.
print("Component transfer functions ready (DLF deferred to the solver).")

In [ ]:
# Noise sources other than the VCO
pn_util = PhaseNoise()

# Reference
reference_phase_noise_obj = PhaseNoise(
    white_noise_100MHz=-500,
    flicker_noise_10kHz=-500,
    noise_floor=-153,
    flicker_noise_distribution_10kHz=-120,
)
reference_phase_noise = reference_phase_noise_obj.get_noise_power(f_offset)

# DAC quantization noise, shaped by the DSM noise transfer function
dac_noise_floor = 1 / 12 / (DSM_FREQ / 2)
dac_noise_ol_linear = dac_noise_floor * np.ones_like(f_offset)
z = np.exp(1j * 2 * np.pi * f_offset * DSM_T)
tf_dsm = 1 - z ** (-DSM_ORDER)
dac_noise_ol_linear_post_dsm = dac_noise_ol_linear * np.abs(tf_dsm) ** 2
# NOTE: as in the 1 GHz notebook, the ZoH is NOT applied here -- it already appears
# inside DigitalLoopFilter's 'ZoH' approximation, so applying it again would double-count.
dac_noise_ol_post_dsm_zoh = pn_util.to_dbc(dac_noise_ol_linear_post_dsm)

# BBPD quantization noise
bbpd_q_noise_power = 4 / 12
bbpd_q_noise_psd = bbpd_q_noise_power / (PLL_FREF / 2)
bbpd_q_noise_linear = bbpd_q_noise_psd * np.ones_like(f_offset)
bbpd_q_noise = pn_util.to_dbc(bbpd_q_noise_linear)

# Reference jitter is loop-invariant -- compute once, outside the solver.
_, reference_jitter = get_jitter(
    f_offset, reference_phase_noise,
    oscillation_frequency=PLL_FOUT,
    integration_bandwidth=INTEGRATION_BANDWIDTH,
)

print(f"BBPD quantization floor : {pn_util.to_dbc(bbpd_q_noise_psd):7.2f} dBc/Hz"
      f"   (1 GHz ref: {pn_util.to_dbc((4/12)/(1e9/2)):.2f})")
print(f"DAC  quantization floor : {pn_util.to_dbc(dac_noise_floor):7.2f} dBc/Hz"
      f"   (1 GHz ref: {pn_util.to_dbc(1/12/(2*1e9/2)):.2f})")
print(f"Unshaped reference jitter at {PLL_FOUT/1e9:.0f} GHz: {reference_jitter:.2f} fs")

## The solver

Three quantities are mutually dependent:

$$K_{bbpd} \;\leftarrow\; \sigma_{jitter} \;\leftarrow\; \text{loop shaping} \;\leftarrow\; (K_p, K_i) \;\leftarrow\; K_{bbpd}$$

Resolved by **alternating iteration with 50 % damping** on the outer jitter loop, with `fsolve`
used only on the inner (well-conditioned) $K_p/K_i$ problem. A single 3-variable `fsolve` is
avoided: inner-convergence noise makes the outer residual non-smooth and it fails to converge.

Undamped direct substitution oscillates; warm-starting $(K_p, K_i)$ from the previous iteration
is what keeps `fsolve` on the correct branch. Typical convergence is ~20–30 iterations.

In [ ]:
def max_feasible_pm(bw, dlf_latency=PLL_DLF_LATENCY_SECONDS):
    '''Largest phase margin the loop can attain at crossover `bw`.

    In the ki -> 0 (pure proportional) limit the loop gain phase is set by the
    integrator, the ZoH half-sample delay, and the transport latencies. The ZoH term
    dominates at a 10 ns sampling period: -18 deg at 10 MHz, vs only -2.5 deg for
    521 ps of DSP latency.
    '''
    s = 2j * np.pi * bw
    tf_zoh = (1 - np.exp(-s * PLL_DIGITAL_T)) / (s * PLL_DIGITAL_T)
    total_latency = dlf_latency + PLL_DIVIDER_LATENCY_SECONDS
    return 180 + np.angle(tf_zoh * np.exp(-total_latency * s) / s) * 180 / np.pi


def solve_loop_filter(kbbpd, bw, pm, guess, dlf_latency=PLL_DLF_LATENCY_SECONDS):
    '''Solve (kp, ki) so the loop gain crosses 0 dB at `bw` with phase margin `pm`.

    Same fsolve pattern as PLL_hybrid_linear_model.ipynb cell 12, restricted to the
    all-digital path. Returns (kp, ki, ok).
    '''
    def loop_filter_equations(vars):
        unknown_kp, unknown_ki = vars
        if unknown_kp <= 0 or unknown_ki <= 0:
            return [1e3, 1e3]                     # penalty: keep the solver in the physical quadrant
        dummy_dlf = components.DigitalLoopFilter(
            kp=unknown_kp, ki=unknown_ki,
            sampling_time=PLL_DIGITAL_T,
            latency=dlf_latency,
            approximation_method='ZoH',
        )
        s = 2j * np.pi * bw
        lg = (kbbpd * dummy_dlf.tf(bw) * (PLL_KDCO / s)
              * np.exp(-PLL_DIVIDER_LATENCY_SECONDS * s) / PLL_DIVIDER_RATIO)
        return [np.abs(lg) - 1,
                np.angle(lg) * 180 / np.pi - pm + 180]

    (kp, ki), _, ier, _ = scipy.optimize.fsolve(
        loop_filter_equations, guess, full_output=True)
    residual = np.max(np.abs(loop_filter_equations((kp, ki))))
    ok = (ier == 1) and (kp > 0) and (ki > 0) and (residual < 1e-6)
    return kp, ki, ok


def solve_operating_point(bw, pm, dlf_latency=PLL_DLF_LATENCY_SECONDS,
                          pm_guard=3.0, relax=0.5, max_iter=200, rel_tol=1e-6,
                          verbose=False):
    '''Self-consistently solve the BBPD gain, loop filter, and output jitter.

    Returns a dict, or None if the (bw, pm) request is infeasible or the solve
    fails to converge. Never returns an unvalidated result -- a silent fsolve
    divergence here yields kp ~ 1e8 and jitter ~ 1e23 fs, so every exit is checked.
    '''
    ceiling = max_feasible_pm(bw, dlf_latency)
    if pm > ceiling - pm_guard:
        if verbose:
            print(f"  INFEASIBLE: PM={pm}deg requested, ceiling is {ceiling:.1f}deg "
                  f"at BW={bw/1e6:.1f} MHz (guard {pm_guard}deg)")
        return None

    jitter = 800.0                                 # fs -- a 100 MHz-scale starting guess
    kp, ki = 0.5, 0.05
    for iteration in range(max_iter):
        sigma_seconds = np.sqrt(jitter ** 2 + reference_jitter ** 2) * 1e-15
        bbpd = components.BangBangPhaseDetector(
            rms_jitter_radians=sigma_seconds * PLL_ANGULAR_FREQUENCY)
        tf_bbpd = bbpd.tf(f_offset)

        kp, ki, ok = solve_loop_filter(tf_bbpd, bw, pm, (kp, ki), dlf_latency)
        if not ok:
            if verbose:
                print(f"  loop-filter solve failed at BW={bw/1e6:.1f} MHz, PM={pm}deg")
            return None

        tf_dlf = components.DigitalLoopFilter(
            kp=kp, ki=ki, sampling_time=PLL_DIGITAL_T,
            latency=dlf_latency, approximation_method='ZoH').tf(f_offset)

        LG = tf_bbpd * tf_dlf * tf_dvco_digital * tf_divider
        sensitivity = 1 / (1 + LG)

        shaping = [
            (dvco_phase_noise,          sensitivity),
            (reference_phase_noise,     tf_bbpd * tf_dlf * tf_dvco_digital * sensitivity),
            (dac_noise_ol_post_dsm_zoh, tf_dvco_digital * sensitivity),
            (bbpd_q_noise,              tf_dlf * tf_dvco_digital * sensitivity),
        ]
        profiles = [get_shaped_phase_noise(f_offset, src, tf) for src, tf in shaping]
        jitters = [get_jitter(f_offset, p, oscillation_frequency=PLL_FOUT,
                              integration_bandwidth=INTEGRATION_BANDWIDTH)[1]
                   for p in profiles]
        total = np.sqrt(sum(j ** 2 for j in jitters))

        if abs(total - jitter) < rel_tol * max(total, 1.0):
            jitter = total
            break
        jitter = jitter + relax * (total - jitter)      # damping is required; relax=1.0 oscillates
    else:
        if verbose:
            print(f"  did not converge in {max_iter} iterations")
        return None

    # Independent post-check: the realized loop gain must actually meet the request.
    realized_pm, realized_bw = get_pm_bw(f_offset, LG)
    if not (abs(realized_pm - pm) < 1.0 and abs(realized_bw - bw) / bw < 0.02):
        if verbose:
            print(f"  post-check FAILED: realized {realized_pm:.1f}deg / "
                  f"{realized_bw/1e6:.2f} MHz vs requested {pm}deg / {bw/1e6:.1f} MHz")
        return None

    total_profile = pn_util.to_dbc(sum(pn_util.to_linear(p) for p in profiles))
    return {
        'total_jitter': jitter,
        'jitters': jitters,                        # [vco, ref, dac_q, bbpd_q]
        'profiles': profiles,
        'total_profile': total_profile,
        'kp': kp, 'ki': ki, 'kbbpd': tf_bbpd,
        'LG': LG,
        'pm': realized_pm, 'bw': realized_bw,
        'iterations': iteration + 1,
        'pm_ceiling': ceiling,
    }


SOURCE_LABELS = ['VCO', 'Reference', 'DAC Quantization', 'BBPD Quantization']
print("Phase-margin ceiling vs bandwidth (ki -> 0 limit):")
for _bw in [8e6, 10e6, 12e6, 14e6]:
    print(f"  {_bw/1e6:5.1f} MHz -> {max_feasible_pm(_bw):5.1f} deg")
print(f"\nThe 1 GHz notebook's PM=77deg is unreachable at every one of these.")

In [ ]:
# Nominal operating point
nominal = solve_operating_point(PLL_BW, PLL_PM, verbose=True)
assert nominal is not None, "nominal operating point failed to solve"

print(f"=== Operating point: BW = {PLL_BW/1e6:.0f} MHz, PM = {PLL_PM} deg ===")
print(f"  converged in {nominal['iterations']} iterations\n")
print(f"  Kp            = {nominal['kp']:.5g}")
print(f"  Ki            = {nominal['ki']:.5g}")
print(f"  Kbbpd         = {nominal['kbbpd']:.4g}")
print(f"  PM ceiling    = {nominal['pm_ceiling']:.1f} deg "
      f"(margin {nominal['pm_ceiling']-PLL_PM:.1f} deg)\n")
print(f"  realized PM   = {nominal['pm']:.1f} deg")
print(f"  realized BW   = {nominal['bw']/1e6:.2f} MHz\n")
print(f"  TOTAL JITTER  = {nominal['total_jitter']:.1f} fs")
for label, j in zip(SOURCE_LABELS, nominal['jitters']):
    share = j ** 2 / nominal['total_jitter'] ** 2 * 100
    print(f"    {label:<20s} {j:7.1f} fs   ({share:4.1f}% of jitter power)")

# Independent verification that fsolve did not silently diverge.
assert abs(nominal['pm'] - PLL_PM) < 0.5, "phase margin post-check failed"
assert abs(nominal['bw'] - PLL_BW) / PLL_BW < 0.01, "bandwidth post-check failed"
print("\n  [ok] get_pm_bw post-check passed -- the solution is real, not a diverged fsolve.")

In [ ]:
# Negative control: ask for more phase margin than the loop physically has.
# Without the guard this returns kp ~ 1e8 and a jitter of ~1e23 fs, and get_pm_bw
# then reports a plausible-looking 348 deg. With the guard it fails cleanly.
print(f"Requesting PM = 75 deg at BW = 10 MHz (ceiling is {max_feasible_pm(10e6):.1f} deg):")
bad = solve_operating_point(10e6, 75, verbose=True)
print(f"  -> returned {bad}")
assert bad is None, "the feasibility guard should have rejected this"
print("\n  [ok] infeasible request rejected cleanly instead of returning garbage.")

In [ ]:
# Closed-loop phase noise, total and per contributor
plot_phase_noise(
    f_offset,
    [nominal['total_profile']] + nominal['profiles'],
    legend_list=['Total'] + SOURCE_LABELS,
    title=f'Closed-loop phase noise -- {PLL_FREF/1e6:.0f} MHz ref, '
          f'{PLL_FOUT/1e9:.0f} GHz out (N={PLL_DIVIDER_RATIO})',
    ylim=(-160, -60),                # default (-180,-80) would clip: in-band total is ~-110
    xlim=(1e3, 4e9),
)

In [ ]:
# Jitter breakdown table and pie chart.
# jitter_breakdown_piechart() treats row 0 as the total and drops it, so Total must come first.
df = jitter_breakdown(
    f_offset,
    [nominal['total_profile']] + nominal['profiles'],
    legend_list=['Total'] + SOURCE_LABELS,
    oscillation_frequency=PLL_FOUT,
    integration_bandwidth=INTEGRATION_BANDWIDTH,
)
display(df)

rss = np.sqrt((df['RJ[fs]'].iloc[1:] ** 2).sum())
total_row = df['RJ[fs]'].iloc[0]
print(f"RSS of contributors : {rss:.1f} fs")
print(f"Total row           : {total_row:.1f} fs")
assert abs(rss - total_row) < 1.0, "breakdown does not RSS to the total"

jitter_breakdown_piechart(df, title='Jitter breakdown -- 100 MHz ref, 8 GHz out')

In [ ]:
# Loop gain Bode plot with crossover and phase margin annotated
LG = nominal['LG']
mag_db = 20 * np.log10(np.abs(LG))
phase_deg = np.unwrap(np.angle(LG)) * 180 / np.pi

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(11, 8))

ax1.semilogx(f_offset, mag_db, 'b-', linewidth=2, label='Loop gain')
ax1.semilogx(f_offset, 20 * np.log10(np.abs(LG / (1 + LG))), 'g--',
             linewidth=1.5, label='Closed loop')
ax1.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax1.axvline(nominal['bw'], color='r', linewidth=1.2, linestyle='--')
ax1.set_ylabel('Magnitude [dB]', fontsize=14)
ax1.set_ylim(-60, 80)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=12)
ax1.set_title(f'Loop gain -- {PLL_FREF/1e6:.0f} MHz ref, N={PLL_DIVIDER_RATIO}', fontsize=15)

ax2.semilogx(f_offset, phase_deg, 'orange', linewidth=2)
ax2.axhline(-180, color='k', linewidth=0.8, linestyle=':')
ax2.axhline(-180 + nominal['pm'], color='r', linewidth=1.0, linestyle='--')
ax2.axvline(nominal['bw'], color='r', linewidth=1.2, linestyle='--')
ax2.set_ylabel('Phase [deg]', fontsize=14)
ax2.set_xlabel('Frequency Offset [Hz]', fontsize=14)
ax2.set_ylim(-270, -60)
ax2.grid(True, alpha=0.3)

ax2.annotate(
    f"BW = {nominal['bw']/1e6:.2f} MHz\nPM = {nominal['pm']:.1f} deg\n"
    f"(ceiling {nominal['pm_ceiling']:.1f} deg)",
    xy=(nominal['bw'], -180 + nominal['pm']), xytext=(0.06, 0.25),
    textcoords='axes fraction', fontsize=13,
    arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
    bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='black'),
)

for ax in (ax1, ax2):
    ax.xaxis.set_major_formatter(FuncFormatter(metric_prefix_formatter))
    ax.set_xlim(1e4, 4e9)

plt.tight_layout()
plt.show()

In [ ]:
# Bandwidth sweep at fixed phase margin.
# Points beyond the feasibility ceiling come back as None and are plotted as gaps.
bw_sweep = np.linspace(2e6, 14e6, 13)
bw_results = [solve_operating_point(bw, PLL_PM) for bw in bw_sweep]

bw_total = np.array([r['total_jitter'] if r else np.nan for r in bw_results])
bw_components = np.array([[r['jitters'][i] if r else np.nan for r in bw_results]
                          for i in range(4)])

n_feasible = int(np.sum(~np.isnan(bw_total)))
print(f"{n_feasible}/{len(bw_sweep)} bandwidth points feasible at PM={PLL_PM} deg")
for bw, r in zip(bw_sweep, bw_results):
    status = f"{r['total_jitter']:7.1f} fs" if r else "INFEASIBLE"
    print(f"  {bw/1e6:5.1f} MHz : {status}")

best = np.nanargmin(bw_total)
print(f"\nMinimum jitter {bw_total[best]:.1f} fs at BW = {bw_sweep[best]/1e6:.1f} MHz")
print(f"Chosen nominal {bw_total[np.argmin(np.abs(bw_sweep-PLL_BW))]:.1f} fs "
      f"at BW = {PLL_BW/1e6:.0f} MHz")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(bw_sweep / 1e6, bw_total, 'k-', marker='o', linewidth=2.5,
        markersize=8, label='Total', zorder=5)
for comp, label, colour in zip(bw_components, SOURCE_LABELS,
                               ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']):
    ax.plot(bw_sweep / 1e6, comp, linestyle='--', marker='s', markersize=5,
            linewidth=1.5, color=colour, alpha=0.85, label=label)

ax.plot(bw_sweep[best] / 1e6, bw_total[best], marker='*', markersize=22,
        color='gold', markeredgecolor='k', zorder=6,
        label=f'Optimum ({bw_sweep[best]/1e6:.0f} MHz)')
ax.axvline(PLL_BW / 1e6, color='purple', linestyle=':', linewidth=2)
ax.annotate(f'nominal\n{PLL_BW/1e6:.0f} MHz', xy=(PLL_BW / 1e6, 250),
            fontsize=12, color='purple', ha='center')

# Shade the region the loop cannot reach at this phase margin
infeasible = bw_sweep[np.isnan(bw_total)]
if len(infeasible):
    ax.axvspan(infeasible.min() / 1e6 - 0.5, bw_sweep.max() / 1e6 + 0.5,
               color='red', alpha=0.12)
    ax.annotate(f'PM={PLL_PM} deg\nunattainable',
                xy=(infeasible.min() / 1e6 + 0.3, 1800),
                fontsize=12, color='darkred')

ax.set_xlabel('Loop Bandwidth [MHz]', fontsize=15)
ax.set_ylabel('RMS Jitter [fs]', fontsize=15)
ax.set_title(f'Jitter vs loop bandwidth at PM = {PLL_PM} deg', fontsize=15)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, ncol=2)
ax.set_ylim(0, 2200)
plt.tight_layout()
plt.show()

In [ ]:
# Phase-margin sweep at the nominal bandwidth
pm_sweep = [45, 50, 55, 60, 65, 70]
print(f"PM sweep at BW = {PLL_BW/1e6:.0f} MHz "
      f"(ceiling = {max_feasible_pm(PLL_BW):.1f} deg):\n")
for pm in pm_sweep:
    r = solve_operating_point(PLL_BW, pm)
    if r:
        print(f"  PM = {pm:2d} deg : {r['total_jitter']:7.1f} fs   "
              f"(kp={r['kp']:.4f}, ki={r['ki']:.5f})")
    else:
        print(f"  PM = {pm:2d} deg : INFEASIBLE (exceeds the {max_feasible_pm(PLL_BW):.1f} deg ceiling)")

## Summary

At **BW = 10 MHz / PM = 60°**, the 100 MHz-reference design achieves **≈ 1.03 ps** RMS output
jitter — roughly 5× worse than the ~200 fs class of the 1 GHz-reference design at the same 8 GHz
output.

**Where the degradation comes from.** No single term dominates; the loop is
VCO / reference / BBPD-quantization co-limited:

| Contributor | Jitter | Share of power | Why it changed |
|---|---|---|---|
| VCO | ~678 fs | ~43 % | Same VCO, but 15× less loop bandwidth to suppress it |
| Reference | ~541 fs | ~27 % | Multiplied by N² — 20 dB worse at N=80 than N=8 |
| BBPD quantization | ~504 fs | ~24 % | Floor rises 10 dB with the 10× lower sampling rate |
| DAC quantization | ~245 fs | ~6 % | Floor rises 10 dB; DSM shaping corner drops 10× |

**The binding constraint is the sampling period, not latency.** At 10 ns the ZoH half-sample
delay costs −18° of phase at 10 MHz, which (a) caps phase margin at 69.5°, making the 1 GHz
design's 77° unreachable, and (b) forces loop bandwidth down to ~10 MHz, which is what lets VCO
noise through. By comparison the 521 ps of DSP latency contributes only −1.9° at this crossover.
That is why this notebook carries no latency sweep: at a bandwidth this low, the latency
sensitivity that drives the 1 GHz notebook's conclusions simply does not bite.

**On bandwidth choice.** Jitter is minimized near 12 MHz (~1010 fs), but that sits only ~5.4°
from the phase-margin ceiling. The 10 MHz nominal gives up ~23 fs (2 %) for a ~9.5° margin —
the better engineering trade. The bandwidth sweep above shows both so the choice stays visible.

**To improve this design,** the highest-leverage moves are a lower-noise VCO (largest single
contributor) or a higher BBPD sampling rate, not latency reduction.

### Not modeled here
The hybrid analog/digital $K_p$ split (cells 8–12 of the 1 GHz notebook, using
`bwrc_data/PFD_CP_noise_2.vcsv`) is out of scope for this pass. `tf_PFD` is defined above so the
analog charge-pump branch can be added without restructuring.